In [7]:
import numpy as np
from pathlib import Path

raster_dir  = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/rasters")
feature_dir = "/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/feature_tensors/"

In [8]:
PATCHES = {
    "Kanto_Japan":      dict(minlat=34.5,  maxlat=37.2,  minlon=138.5, maxlon=141.5),
    "Tohoku_Japan":     dict(minlat=37.5,  maxlat=40.5,  minlon=140.5, maxlon=143.5),
    "Central_Chile":    dict(minlat=-36.5, maxlat=-33.5, minlon=-72.5, maxlon=-69.5),
    "Central_Turkey":   dict(minlat=36.5,  maxlat=39.0,  minlon=35.5,  maxlon=39.0),
    "Central_Nepal":    dict(minlat=27.0,  maxlat=29.7,  minlon=83.5,  maxlon=86.5),
    "North_Island_NZ":  dict(minlat=-40.5, maxlat=-37.5, minlon=174.5, maxlon=178.0),
    "Sumatra":          dict(minlat=-5.5,  maxlat=-2.0,  minlon=100.5, maxlon=104.5),
    "Kutch_India":      dict(minlat=21.5,  maxlat=24.5,  minlon=68.5,  maxlon=72.0),
    "Sichuan_China":    dict(minlat=29.5,  maxlat=32.5,  minlon=102.0, maxlon=105.5),
    "W_Australia":      dict(minlat=-32.0, maxlat=-29.0, minlon=117.0, maxlon=120.5),
    "S_Norway":         dict(minlat=58.5,  maxlat=61.5,  minlon=5.0,   maxlon=9.0),
    "Ordos_China":      dict(minlat=37.0,  maxlat=40.0,  minlon=107.5, maxlon=111.0),
}

RES = 0.1

# Feature definitions — name, filename suffix, normalisation strategy
# norm: 'standard' = (x-mean)/std, 'minmax' = (x-min)/(max-min), 'none' = raw
FEATURES = [
    ('vs30',           'vs30',           'standard'),
    ('elevation',      'dem',            'standard'),
    ('slope',          'slope',          'standard'),
    ('roughness',      'roughness',      'standard'),
    ('sediment_km',    'sediment',       'standard'),
    ('crustal_km',     'crustal',        'standard'),
    ('dist_fault_km',  'dist_fault',     'standard'),
    ('fault_density',  'fault_density',  'standard'),
    ('fault_slip',     'fault_slip',     'none'),      # categorical — no normalisation
    ('heat_flow',      'heatflow',       'standard'),
    ('stress_azi',     'stress_azi',     'none'),      # circular — handle separately
    ('stress_regime',  'stress_regime',  'none'),      # categorical
]

FEATURE_NAMES = [f[0] for f in FEATURES]

In [9]:
# We normalise globally across all patches — not per patch
# This ensures the same scale is used everywhere
# First pass: collect all valid values per feature for global stats
print("Computing global normalisation statistics...")
global_vals = {fname: [] for fname, _, norm in FEATURES if norm == 'standard'}

for patch_name in PATCHES:
    for fname, fsuffix, norm in FEATURES:
        if norm != 'standard':
            continue
        fpath = raster_dir / f"{patch_name}_{fsuffix}.npy"
        if fpath.exists():
            arr = np.load(fpath).astype(float)
            valid = arr[~np.isnan(arr)].ravel()
            global_vals[fname].extend(valid.tolist())

# Compute global mean and std
global_stats = {}
for fname, vals in global_vals.items():
    arr = np.array(vals)
    global_stats[fname] = {
        'mean': float(np.mean(arr)),
        'std':  float(np.std(arr)),
        'min':  float(np.min(arr)),
        'max':  float(np.max(arr)),
    }
    print(f"  {fname:<20} mean={global_stats[fname]['mean']:>8.2f}  "
          f"std={global_stats[fname]['std']:>8.2f}  "
          f"min={global_stats[fname]['min']:>8.2f}  "
          f"max={global_stats[fname]['max']:>8.2f}")

# Save global stats for use during inference
import json
with open(feature_dir + "global_norm_stats.json", "w") as f:
    json.dump(global_stats, f, indent=2)

# ── Second pass: stack and normalise per patch ────────────────────────────────
print(f"\n{'Patch':<25} {'Shape':>15} {'NaN cells':>10} {'NaN%':>8}")
print("-" * 65)

for patch_name, b in PATCHES.items():
    lons = np.arange(b['minlon'] + RES/2, b['maxlon'], RES)
    lats = np.arange(b['minlat'] + RES/2, b['maxlat'], RES)
    n_lat, n_lon = len(lats), len(lons)
    n_features   = len(FEATURES)

    # Stack: shape (n_lat, n_lon, n_features)
    tensor = np.full((n_lat, n_lon, n_features), np.nan)

    for fi, (fname, fsuffix, norm) in enumerate(FEATURES):
        fpath = raster_dir / f"{patch_name}_{fsuffix}.npy"
        if not fpath.exists():
            print(f"  WARNING: {fpath} not found — leaving as NaN")
            continue

        arr = np.load(fpath).astype(float)

        # Ensure shape matches
        if arr.shape != (n_lat, n_lon):
            print(f"  WARNING: {patch_name} {fname} shape {arr.shape} "
                  f"!= expected {(n_lat, n_lon)} — skipping")
            continue

        # Normalise
        if norm == 'standard':
            mean = global_stats[fname]['mean']
            std  = global_stats[fname]['std']
            arr  = (arr - mean) / (std + 1e-8)

        elif norm == 'none':
            pass  # keep raw values

        tensor[:, :, fi] = arr

    # Save tensor and coordinate arrays
    np.save(feature_dir + f"{patch_name}_features.npy", tensor)
    np.save(feature_dir + f"{patch_name}_lons.npy",     lons)
    np.save(feature_dir + f"{patch_name}_lats.npy",     lats)

    # Report NaN statistics
    nan_cells = np.isnan(tensor).any(axis=-1).sum()
    nan_pct   = 100 * nan_cells / (n_lat * n_lon)

    print(f"{patch_name:<25} "
          f"{str(tensor.shape):>15} "
          f"{nan_cells:>10} "
          f"{nan_pct:>7.1f}%")

# Save feature name index
with open(feature_dir + "feature_names.json", "w") as f:
    json.dump(FEATURE_NAMES, f, indent=2)

print(f"\nFeature names: {FEATURE_NAMES}")
print(f"\nSaved feature tensors to data/feature_tensors/")
print(f"  Files per patch: {{patch}}_features.npy  shape (n_lat, n_lon, {len(FEATURES)})")
print(f"  Also saved: global_norm_stats.json, feature_names.json")

Computing global normalisation statistics...
  vs30                 mean=  566.67  std=  214.39  min=  159.29  max= 1001.98
  elevation            mean=  876.42  std= 1176.35  min=   -2.23  max= 6173.92
  slope                mean=    0.72  std=    1.01  min=    0.00  max=    9.54
  roughness            mean=  117.92  std=  150.98  min=    0.00  max= 1376.14
  sediment_km          mean=    1.26  std=    1.63  min=    0.00  max=    8.81
  crustal_km           mean=   35.95  std=    9.85  min=    7.83  max=   71.48
  dist_fault_km        mean=  161.35  std=  283.34  min=    0.14  max=  999.00
  fault_density        mean=    0.28  std=    0.63  min=    0.00  max=    2.50
  heat_flow            mean=   74.47  std=   28.93  min=   30.19  max=  231.72

Patch                               Shape  NaN cells     NaN%
-----------------------------------------------------------------
Kanto_Japan                  (27, 30, 12)        214    26.4%
Tohoku_Japan                 (30, 30, 12)        398 

In [10]:
feature_dir = Path("/Users/rahulravi23/Desktop/Work/seismic_hazard_modelling/seismic_hazard_modelling/data/feature_tensors")

with open(feature_dir / "feature_names.json") as f:
    feature_names = json.load(f)

print(f"Features ({len(feature_names)}): {feature_names}\n")

print(f"{'Patch':<25} {'Shape':>15} "
      + "  ".join(f"{n[:6]:>8}" for n in feature_names[:6]))
print("-" * 100)

patches = [
    "Kanto_Japan", "Tohoku_Japan", "Central_Chile", "Central_Turkey",
    "Central_Nepal", "North_Island_NZ", "Sumatra", "Kutch_India",
    "Sichuan_China", "W_Australia", "S_Norway", "Ordos_China"
]

for patch_name in patches:
    t = np.load(feature_dir / f"{patch_name}_features.npy")
    # Mean of each feature across valid cells
    means = [np.nanmean(t[:,:,i]) for i in range(t.shape[2])]
    print(f"{patch_name:<25} {str(t.shape):>15}  "
          + "  ".join(f"{m:>8.2f}" for m in means[:6]))

Features (12): ['vs30', 'elevation', 'slope', 'roughness', 'sediment_km', 'crustal_km', 'dist_fault_km', 'fault_density', 'fault_slip', 'heat_flow', 'stress_azi', 'stress_regime']

Patch                               Shape     vs30    elevat     slope    roughn    sedime    crusta
----------------------------------------------------------------------------------------------------
Kanto_Japan                  (27, 30, 12)     -0.16     -0.48     -0.00     -0.06     -0.23     -1.00
Tohoku_Japan                 (30, 30, 12)      0.09     -0.56     -0.17     -0.27     -0.36     -1.32
Central_Chile                (30, 30, 12)      0.26      0.38      0.56      0.62     -0.53      0.56
Central_Turkey               (25, 35, 12)     -0.21      0.13      0.24      0.28     -0.38      0.12
Central_Nepal                (27, 30, 12)      0.58      1.93      1.27      1.43     -0.25      1.87
North_Island_NZ              (30, 35, 12)      0.93     -0.50     -0.19     -0.21      0.11     -0.55
Sumat